# Plot Checkpoint Predictions - v4

Select a trained checkpoint, one session date, and a ticker list. The notebook rebuilds chronological 1-minute windows from the provider bars, runs one-step inference, and plots predicted close vs target close for each ticker.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v4"
MODEL_ROOT = Path(r"D:\\TradingData\\quant-research-workbench\\market_data\\models\\inhouse_transformer") / MODEL_VERSION

# Paste a trained v4 checkpoint path, or leave empty to auto-load the latest v4 last.pt/best.pt.
CHECKPOINT_PATH = ""

# Inference selection.
SESSION_DATE = "2024-01-22"
TICKERS = ("VFC", "USO", "CADL")

# Runtime controls.
WORKSPACE = Path(r"D:\\TradingCodes\\quant-research-workbench")
DEVICE = "cuda"
USE_AMP = True
INFERENCE_BATCH_SIZE = 2048

# 0 means plot every valid chronological window for each ticker.
MAX_POINTS_PER_TICKER = 0

In [ ]:
import sys
from dataclasses import fields
from datetime import datetime

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch

if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.inhouse_transformer.v4.config import DataConfig, ExperimentConfig, ModelConfig, TrainConfig
from research.inhouse_transformer.v4.data import available_sessions
from research.inhouse_transformer.v4 import train as train_mod

train_mod.load_torch_stack()

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (18, 8),
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "legend.fontsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "lines.linewidth": 2.0,
    }
)

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)

In [ ]:
def dataclass_from_dict(cls, payload, tuple_fields=()):
    allowed = {field.name for field in fields(cls)}
    values = {key: value for key, value in payload.items() if key in allowed}
    if cls is DataConfig and "processed_root" in values:
        values["processed_root"] = Path(values["processed_root"])
    for name in tuple_fields:
        if name in values:
            values[name] = tuple(values[name])
    return cls(**values)


def latest_checkpoint(model_root):
    candidates = list(model_root.glob("*/last.pt")) + list(model_root.glob("*/best.pt"))
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found under {model_root}. Set CHECKPOINT_PATH manually.")
    return max(candidates, key=lambda path: path.stat().st_mtime)

checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(MODEL_ROOT)
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
if not TICKERS:
    raise ValueError("Set TICKERS to at least one ticker symbol.")

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location="cpu")
checkpoint_config = checkpoint.get("config")
if not checkpoint_config:
    raise KeyError("Checkpoint does not contain a config payload, so model/data shape cannot be restored safely.")
checkpoint_version = checkpoint_config.get("experiment_version")
if checkpoint_version and checkpoint_version != MODEL_VERSION:
    raise ValueError(f"Checkpoint version {checkpoint_version!r} does not match notebook version {MODEL_VERSION!r}.")

data_config = dataclass_from_dict(
    DataConfig,
    checkpoint_config.get("data", {}),
    tuple_fields=("target_columns", "input_feature_columns", "time_feature_columns", "tickers"),
)
model_config = dataclass_from_dict(ModelConfig, checkpoint_config.get("model", {}))
train_config = dataclass_from_dict(TrainConfig, checkpoint_config.get("train", {}))

# Keep checkpoint model/data shape, but override the plotted session/tickers/runtime.
data_config.train_start_date = SESSION_DATE
data_config.train_end_date = SESSION_DATE
data_config.validation_start_date = SESSION_DATE
data_config.validation_end_date = SESSION_DATE
data_config.test_start_date = SESSION_DATE
data_config.test_end_date = SESSION_DATE
data_config.tickers = tuple(ticker.upper() for ticker in TICKERS)
data_config.max_tickers = len(data_config.tickers)
train_config.batch_size = INFERENCE_BATCH_SIZE
train_config.amp = USE_AMP

config = ExperimentConfig(data=data_config, model=model_config, train=train_config)

if "close" not in config.data.target_columns:
    raise ValueError(f"This notebook plots close, but checkpoint target_columns={config.data.target_columns}")

sessions = available_sessions(config.data.processed_root, SESSION_DATE, SESSION_DATE)
print(f"checkpoint={checkpoint_path}")
print(f"checkpoint_step={checkpoint.get('step')} best_score={checkpoint.get('best_score')}")
print(f"device={device} amp={config.train.amp and device.type == 'cuda'} batch_size={config.train.batch_size}")
print(f"session={sessions[0]} tickers={config.data.tickers}")
print(f"context={config.data.context_length} horizon={config.data.horizon} targets={config.data.target_columns}")
print(f"target_mode={config.data.target_mode} input_normalization={config.data.input_normalization}")
print(f"features={len(config.data.input_feature_columns)} time_features={len(config.data.time_feature_columns)}")

In [ ]:
model = train_mod.FeatureTemporalTransformer(
    feature_count=len(config.data.input_feature_columns),
    time_feature_count=len(config.data.time_feature_columns),
    context_length=config.data.context_length,
    horizon=config.data.horizon,
    target_count=len(config.data.target_columns),
    config=config.model,
).to(device)
model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

param_count = sum(parameter.numel() for parameter in model.parameters())
print(f"loaded model parameters={param_count:,}")

In [ ]:
rows_by_ticker = train_mod.infer_session_timeline_predictions(
    model=model,
    config=config,
    session=sessions[0],
    tickers=config.data.tickers,
    device=device,
    max_points_per_ticker=MAX_POINTS_PER_TICKER,
)

if not rows_by_ticker:
    raise RuntimeError("No valid prediction rows were created. Check date, tickers, context length, and horizon.")

summary_rows = []
for ticker, rows in rows_by_ticker.items():
    target_bps = np.array([float(row["target_bps"]) for row in rows])
    prediction_bps = np.array([float(row["prediction_bps"]) for row in rows])
    target_close = np.array([float(row["target_close"]) for row in rows])
    prediction_close = np.array([float(row["prediction_close"]) for row in rows])
    error_bps = prediction_bps - target_bps
    direction_correct = (prediction_bps > 0.0) == (target_bps > 0.0)
    summary_rows.append(
        {
            "ticker": ticker,
            "windows": len(rows),
            "target_start": rows[0]["target_time"],
            "target_end": rows[-1]["target_time"],
            "mae_price": float(np.mean(np.abs(prediction_close - target_close))),
            "mae_bps": float(np.mean(np.abs(error_bps))),
            "rmse_bps": float(np.sqrt(np.mean(error_bps ** 2))),
            "bias_bps": float(np.mean(error_bps)),
            "direction_accuracy_pct": float(np.mean(direction_correct) * 100.0),
        }
    )

summary = pl.DataFrame(summary_rows)
summary

In [ ]:
def plot_ticker_predictions(ticker, rows):
    rows = sorted(rows, key=lambda row: int(row["bar_index"]))
    times = [datetime.fromisoformat(str(row["target_time"])) for row in rows]
    target_close = np.array([float(row["target_close"]) for row in rows])
    prediction_close = np.array([float(row["prediction_close"]) for row in rows])
    target_bps = np.array([float(row["target_bps"]) for row in rows])
    prediction_bps = np.array([float(row["prediction_bps"]) for row in rows])
    error_bps = prediction_bps - target_bps
    direction_correct = (prediction_bps > 0.0) == (target_bps > 0.0)

    mae_bps = float(np.mean(np.abs(error_bps)))
    rmse_bps = float(np.sqrt(np.mean(error_bps ** 2)))
    bias_bps = float(np.mean(error_bps))
    dir_acc = float(np.mean(direction_correct) * 100.0)

    fig, (price_ax, error_ax) = plt.subplots(
        2,
        1,
        figsize=(18, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.1], "hspace": 0.08},
    )
    fig.suptitle(
        f"{ticker} | {SESSION_DATE} | h1 close prediction vs target | "
        f"MAE {mae_bps:.2f} bps | RMSE {rmse_bps:.2f} bps | bias {bias_bps:.2f} bps | dir {dir_acc:.1f}%",
        y=0.98,
        fontweight="bold",
    )

    price_ax.plot(times, target_close, color="#111827", label="target close", linewidth=2.3)
    price_ax.plot(
        times,
        prediction_close,
        color="#f97416c7",
        label="prediction close",
        linewidth=1.0,

    )
    price_ax.set_ylabel("close price")
    price_ax.legend(loc="upper left", frameon=True)
    price_ax.grid(True, color="#e5e7eb", linewidth=0.8)

    error_ax.axhline(0.0, color="#374151", linewidth=1.0)
    error_ax.fill_between(times, error_bps, 0.0, color="#f97316", alpha=0.18)
    error_ax.plot(times, error_bps, color="#ea580c", linewidth=1.3, label="prediction error bps")
    error_ax.set_ylabel("error bps")
    error_ax.set_xlabel("target bar time")
    error_ax.grid(True, color="#e5e7eb", linewidth=0.8)

    locator = mdates.AutoDateLocator(minticks=5, maxticks=12)
    formatter = mdates.ConciseDateFormatter(locator)
    error_ax.xaxis.set_major_locator(locator)
    error_ax.xaxis.set_major_formatter(formatter)
    fig.autofmt_xdate()
    plt.show()


for ticker, rows in rows_by_ticker.items():
    plot_ticker_predictions(ticker, rows)

In [ ]:
# Optional: inspect the exact plotted rows as a table.
all_rows = []
for ticker, rows in rows_by_ticker.items():
    for row in rows:
        all_rows.append(row)

pl.DataFrame(all_rows).select(
    "ticker",
    "bar_index",
    "target_time",
    "current_close",
    "target_close",
    "prediction_close",
    "target_bps",
    "prediction_bps",
    "abs_error_bps",
).head(50)